## NumPy 矩陣轉換與 Pandas 日誌清洗

- 目標：掌握 NumPy 廣播機制在晶圓像素處理上的應用，精通 **Pandas 高階函數**進行 **MES 日誌清洗與新產品引進（NPI）關鍵品質特性（CTQ）資料的整併**，並使用 **Matplotlib/Seaborn 將數據視覺化**。


### 1. NumPy 矩陣廣播與晶圓圖 (Wafer Map) 張量轉換

- 概念：
    - 廣播：形狀不同的陣列在特定規則下可以直接做逐元素運算（Element-wise operation），不需要手動迴圈展開，NumPy 會自動把較小的陣列「延展」排列到更大的形狀。
    - 廣播規則：從最後一個維度開始比對，兩個維度要其中一個是1，才能廣播，例如 (H, W) 的晶圓圖要疊上 (3,) 的顏色權重時，需要先重塑成 (1, 1, 3) 才能對齊。
    - 維度補充：np.newaxis 或 reshape 常用來把二維 Bin Map(H, W) 補充成三維 (H, W, 1)，方便後續做通道複製或 RGB 上色。
    - 向量化運算 (Vectorization) vs Python 迴圈：NumPy 底層是 C 實作的連續記憶體配置（Contiguous Memory Allocation），速度遠快於 Python 逐格迴圈，對大晶圓矩陣特別重要。
    - 應用連結：Bin Code（如 Pass=1, Fail=2, Edge=3）轉換成顏色遮罩，本質就是用布林索引器（array == 某值）配合廣播，把符合條件的位置賦予特定的 RGB 值。


- 實作：
    - **晶圓圖**通常會被表達為**二維矩陣（Height \(\times \) Width）**，每個位置代表一個晶粒（Die）的測試結果（Bin Code）。**為了將其輸入到深度學習模型（如 CNN）**，我們通常**需要將其轉化為三維張量（\(H \times W \times 3\)，即 RGB 三通道）**，並透過**矩陣廣播（Broadcasting）為特定缺陷類型進行顏色增強或遮罩處理**。


In [ ]:
import numpy as np

# 模擬一個 5x5 的單層晶圓地圖矩陣 (0: Pass, 1: Edge Defect, 2: Scratch)
wafer_2d = np.array([
    [0, 1, 0, 1, 0],
    [1, 2, 1, 2, 1],
    [0, 1, 0, 1, 0],
    [1, 2, 1, 2, 1],
    [0, 1, 0, 1, 0]
])
print("原始 2D 晶圓矩陣形狀:", wafer_2d.shape)

# 將 2D 矩陣轉換為 3D 張量通道的基礎 (透過擴展維度為 H x W x 1)
wafer_3d_base = np.expand_dims(wafer_2d, axis=-1)

# 運用 NumPy 廣播機制：與一個 (1, 1, 3) 的色彩權重矩陣相乘，快速生成 RGB 三通道張量
# 假設我們想讓不同的缺陷在不同通道放大：通道 0 強調邊緣缺陷(1)，通道 1 強調刮痕(2)
color_weights = np.array([[[255, 100, 50]]], dtype=np.uint8) # 形狀為 (1, 1, 3)

# 廣播運算：(H, W, 1) * (1, 1, 3) -> 變成 (H, W, 3)
wafer_rgb_tensor = wafer_3d_base * color_weights

print("廣播後 3D RGB 張量形狀:", wafer_rgb_tensor.shape)
print("晶圓中心點 (座標 2,2) 轉換後的 RGB 像素值:", wafer_rgb_tensor[2, 2])

### 2. Pandas 實戰：MES Log 清洗與 NPI CTQ 數據整併

- 概念：
    - DataFrame 核心二維概念：可以想成「帶有欄位名稱與索引的表格」，其實每一欄其實都是 NumPy 數組，因此欄與欄位之間可以是不同的型態。
    - 去重（Deduplication）：drop_duplicates() 依欄指定位元（如 Lot ID + Wafer ID + 時間）決定重複列，MES 系統常因網路重傳造成同一筆資料寫入兩次。
    - 合併（merge）：
        - inner：只保留兩表（或雙方）都有對應鍵值的資料（找出兩個表格都有記錄的批次）。
        - left：保留左表全部數據，右表沒有對應到的欄位補 NaN（常用在「以測試數據為主，補充 CTQ 資訊，缺少的先留空」）。
        - outer：兩邊資料都保留，以便抓出「其中哪一批只出現在其中一張表」的資料落差。
    - 分組聚合（groupby）：先依鍵值（如工具 ID）分組，再對每組分組用聚合函數（mean、std、count），是良率趨勢依機台拆解分析的基礎。
    - 合併前務必確認兩表的鍵值欄位型態一致（例如一邊是字符串"25"、一邊是整數25），不一致會導致合併後全部對不上、結果無聲無息地變空。


- 實作：
    - 我們需要將製造執行系統（MES）產生的髒日誌進行去重與清洗，並透過批號（Lot ID）與晶圓號（Wafer ID）與新產品引進（NPI）階段記錄的關鍵品質特性（CTQ，如測試電壓、電阻等參數）**資料表進行合併（Merge）**，最後依據製程機台（Tool ID）進行**分組（Groupby）分析**。


In [ ]:
import pandas as pd

# 模擬 MES 生產日誌 (包含重置、缺失值與髒資料)
mes_log_data = {
    "lot_id": ["LOT_A01", "LOT_A01", "LOT_A02", "LOT_A02", "LOT_A01"],
    "wafer_id": ["WAFER_01", "WAFER_02", "WAFER_03", "WAFER_04", "WAFER_05"],
    "tool_id": ["TOOL_01", "TOOL_01", "TOOL_02", "TOOL_02", "TOOL_01"],
    "status": ["PASS", "PASS", "FAIL", "PASS", None],  # 包含重複與缺失值
}
df_mes = pd.DataFrame(mes_log_data)

# 清洗步驟：去重 (Drop Duplicates) 與 填補缺失值 (Fillna)
print(">>> 執行 MES Log 清洗...")
df_mes_clean = df_mes.drop_duplicates().dropna(subset=["status"]).copy()
# 對剩餘的潛在缺失或異常文字進行標準化處理
df_mes_clean["status"] = df_mes_clean["status"].str.upper()

# 模擬 NPI 實驗室的 CTQ 參數測試資料
ctq_data = {
    "lot_id": ["LOT_A01", "LOT_A01", "LOT_A02", "LOT_A02"],
    "wafer_id": ["WAFER_01", "WAFER_02", "WAFER_03", "WAFER_04"],
    "bandwidth_ghz": [28.4, 27.9, 22.1, 28.1],  # 頻寬測試數據
    "leakage_current_ma": [0.05, 0.06, 0.45, 0.04],
}
df_ctq = pd.DataFrame(ctq_data)

# 數據整併 (Merge)：使用多欄位作為複合主鍵 (Composite Key) 進行對齊
print(">>> 執行 MES 與 NPI CTQ 數據整併...")
df_merged = pd.merge(df_mes_clean, df_ctq, on=["lot_id", "wafer_id"], how="inner")
print(df_merged)

# 高階分組計算 (Groupby & Pivot Table)：計算各機台的平均頻寬與不良率
print("\n>>> 分析各機台 (Tool ID) 的統計指標：")
tool_summary = (
    df_merged.groupby("tool_id")
    .agg(
        avg_bandwidth=("bandwidth_ghz", "mean"),
        max_leakage=("leakage_current_ma", "max"),
        wafer_count=("wafer_id", "count"),
    )
    .reset_index()
)
print(tool_summary)

### 3. 多執行緒與多進程平行化處理

- 概念：
    - GIL（全域直譯器鎖定）：CPython 同樣時間只允許一條執行緒執行 Python 字節碼，因此純計算密集（CPU-bound）的任務使用多執行緒不會真正並行加速。
    - I/O-bound 與 CPU-bound 的判斷：
        - I/O-bound（等待網路、等待磁盤讀寫器）：執行緒在等待時會釋放 GIL，接下來ThreadPoolExecutor 有效，例如同時對多台機台發送 SCPI 指令並等待應答。
        - CPU-bound（大量數值計算、影像處理）：需要 ProcessPoolExecutor，每個流程都有獨立的直譯器和記憶體空間，才能真正利用多核心。
    - concurrent.futures 的統一概念：ThreadPoolExecutor 與 ProcessPoolExecutor 消耗相同的 API （submit、map、as_completed），切換執行策略時，幾乎不需要改動核心的業務邏輯程式碼。
    - 多進程的額外成本：跨進程沒有共享記憶體，資料提交需要序列化（pickle），因此如果任務本身速度、資料量卻很大，開多進程反而可能比單執行緒慢。


- 實作：
    - 當你需要同時清洗幾十萬行的日誌，或是對大量晶圓地圖（Wafer Map）進行平行運算與特徵提取時，單核心 Python 往往會跑太慢。
- 解析：
    - ThreadPoolExecutor：適用於 I/O 密集任務（例如同時讀取多個遠端資料庫或大量檔案）。
    - ProcessPoolExecutor：適用於 CPU 密集任務（例如平行執行機器學習推理、影像處理或大量數據運算）。


### 4. 數據視覺化：良率趨勢圖與 Wafer Bin Map 佈局

- 概念：
    - Matplotlib 的物件引導面：暗示用 fig, ax = plt.subplots() 明確的方式獲得圖形與軸物件操作，而不是全域的 plt.plot()，在多子圖（如同時繪製良率趨勢與 Bin Map）時不會比較互相干擾。
    - 折線圖（良率趨勢）：時間軸格式化、控制上下限的水平參考線、瞄準線，這些元素讓工程師快速判斷是否處於狀態。
    - 熱圖（Wafer Bin Map）：seaborn.heatmap() 或 imshow() 將二維矩陣直接映射成顏色深淺，需注意色階（colormap）選擇 — 類別型 Bin Code（Pass/Fail/不同缺陷類別）適合用離散色階，不適合用連續漸進層色，否則視覺上會誤差成「數值大小」或「類別」。
    - 座標系統對應：晶圓圖有實體圓形邊界，繪製時常需要增加一個圓形遮光罩，把矩陣中「圓外」的無效格子顯示為空白，避免誤判為缺陷。
- 實作：將整併後的數據，繪製成每日的**良率趨勢圖（Yield Trend）**以及用**熱圖（Heatmap）呈現的晶圓點位圖（Wafer Bin Map）**，方便工程師一眼看出是否有**邊緣缺陷（Edge Ring）或局部群聚（Cluster）異常**。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 設定中文與負號顯示（防止圖表亂碼，視作業系統可能需要調整字體）
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False

# --- 繪製良率趨勢圖 (Yield Trend) ---
dates = pd.date_range(start="2026-08-01", periods=7)
yield_rates = [
    98.2,
    97.9,
    94.1,
    98.0,
    98.5,
    91.2,
    98.1,
]  # 觀察第 3 天與第 6 天的良率下墜

plt.figure(figsize=(8, 3.5))
plt.plot(dates, yield_rates, marker="o", color="b", linestyle="--", linewidth=2)
plt.axhline(y=95.0, color="r", linestyle=":", label="良率警報線 (95%)")
plt.title("NPI 專案：每日測試良率趨勢圖 (Yield Trend)", fontsize=12)
plt.xlabel("測試日期")
plt.ylabel("良率 (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- 繪製晶圓圖點位分佈 (Wafer Bin Map Heatmap) ---
# 建立一個較大的模擬 10x10 晶圓測試結果（數值代表 Bin Code）
np.random.seed(42)
mock_wafer_grid = np.zeros((10, 10))
# 模擬中心與邊緣有一些不合規的 Bin 2 (例如電性測試失敗)
mock_wafer_grid[2:4, 3:5] = 2  # 模擬群聚缺陷 (Cluster)
mock_wafer_grid[np.random.choice(10, 5), np.random.choice(10, 5)] = 1  # 隨機缺陷

plt.figure(figsize=(6, 5))
sns.heatmap(
    mock_wafer_grid, annot=True, cmap="YlOrRd", cbar_kws={"label": "Bin Code 型態"}
)
plt.title("自動生成：Wafer Bin Map 缺陷分佈圖", fontsize=12)
plt.xlabel("晶圓 X 座標")
plt.ylabel("晶圓 Y 座標")
plt.tight_layout()
plt.show()

- 總結：在處理大量晶圓地圖資料時，為了不浪費效能，我會避免使用 Python 的 for 迴圈，而是善用 NumPy 的廣播機制 (Broadcasting) 與向量化運算，將 2D 的 Bin Map 快速映射轉換為 3D RGB 張量送進 CNN 模型。另外，在生產端往往會有 MES Log 的重複上報或缺失值，我會透過 Pandas 的 drop_duplicates 與多鍵 merge 函數，將髒日誌與 NPI 的 CTQ 電性參數精準對齊，並用 Seaborn 繪製 Wafer Bin Map 熱圖，以便在面試與產線報告中，直觀地解釋 Cluster 或 Edge 缺陷的物理成因。
